## Opening files

In [1]:
import pandas as pd

In [2]:
query_df = pd.read_csv("../data/requik/queries/queries.tsv", sep="\t", header=None)
query_df.columns = ["qid", "query"]
query_df["query"] = ["".join([x if x.isalnum() else " " for x in query]) for query in query_df["query"]] # removing punctuations
query_df["qid"] = query_df["qid"].astype(str)
query_df.to_csv("../data/requik/inputs/topics.csv", index=False)
query_df.head()

,qid,query
0,RQq1,when was the last time capulin erupted
1,RQq2,What year did the Capulin Volcano in New Mexic...
2,RQq3,What year did the Capulin Volcano Last errupt
3,RQq4,In Arizona what dessert is the Organ Pipe Cact...
4,RQq5,In Arizona what dessert is the Organ Pipe Cact...


In [13]:
qrels_df = pd.read_csv("../data/requik/qrels/qrels-extended-relevance.tsv", sep="\t", header=None)
qrels_df.columns = ["qid", "q0", "docno", "relevance"]
qrels_df["relevance"] = [int(rel==3) for rel in qrels_df["relevance"]]
qrels_df.to_csv("../data/requik/qrels/qrels-relevance-binary.csv", index=False)
qrels_df.head()

,qid,q0,docno,relevance
0,RQq1,0,RQ_d1,1
1,RQq1,0,RQ_d2,0
2,RQq1,0,RQ_d3,0
3,RQq1,0,RQ_d4,0
4,RQq1,0,RQ_d5,0


## Adding more document features

### readability

In [21]:
import json
import pandas as pd
from spache_allen.formula import spache_allen
from tqdm import tqdm

lines = []
with open(r'../data/requik/corpus/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
corpus = pd.DataFrame(line_dicts)
corpus.drop_duplicates(inplace=True,ignore_index=True)
corpus.columns = ["docno", "main_content"]
corpus["readability"] = [spache_allen(text) for text in tqdm(corpus["main_content"], total=len(corpus))]
corpus.to_csv("../data/requik/inputs/documents.csv", index=False)
print(len(corpus))
corpus.head()


100%|██████████| 317/317 [00:03<00:00, 84.09it/s]

317


,docno,main_content,readability
0,RQ_d1,When was the last time the capulin volcano nat...,4.418878
1,RQ_d2,Frequently Asked Questions - Capulin Volcano N...,3.050643
2,RQ_d3,When was the last time the capulin volcano nat...,4.418878
3,RQ_d4,Capulin Volcano National Monument - Britannica...,5.334789
4,RQ_d5,Capulin Volcano National Monument - Wikipedia....,4.335093


### objectivity

In [1]:
import json
import pandas as pd
from utils import get_obj_likelihood
from tqdm import tqdm

lines = []
with open(r'../data/requik/corpus/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
corpus = pd.DataFrame(line_dicts)
corpus.drop_duplicates(inplace=True,ignore_index=True)
corpus.columns = ["docno", "main_content"]
corpus["obj_prob"] = get_obj_likelihood(list(corpus["main_content"]))
corpus = corpus[["docno","obj_prob"]]

corpus_df = pd.read_csv("../data/requik/inputs/documents.csv")
corpus_df = corpus_df.merge(corpus, on="docno")
corpus_df.to_csv("../data/requik/inputs/documents.csv", index=False)
corpus_df.head()

/home/hchakrabar/miniconda3/envs/pt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-29 23:00:36.827092: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-29 23:00:36.875431: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-29 23:00:38.005862: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You

,docno,main_content,readability,edu_val,obj_prob
0,RQ_d1,When was the last time the capulin volcano nat...,4.418878,2,0.433879
1,RQ_d2,Frequently Asked Questions - Capulin Volcano N...,3.050643,2,0.792916
2,RQ_d3,When was the last time the capulin volcano nat...,4.418878,2,0.443689
3,RQ_d4,Capulin Volcano National Monument - Britannica...,5.334789,3,0.940539
4,RQ_d5,Capulin Volcano National Monument - Wikipedia....,4.335093,3,0.866994


### educational alignment

In [10]:
import json
import pandas as pd
from tqdm import tqdm
from utils import get_edu_value

lines = []
with open(r'../data/requik/corpus/documents.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
corpus = pd.DataFrame(line_dicts)
corpus.drop_duplicates(inplace=True,ignore_index=True)
corpus.columns = ["docno", "main_content"]
corpus["edu_val"] = [get_edu_value(text) for text in tqdm(corpus["main_content"], total=len(corpus))]

corpus = corpus[["docno","edu_val"]]

corpus_df = pd.read_csv("../data/requik/inputs/documents.csv")
corpus_df = corpus_df.merge(corpus, on="docno")
corpus_df.to_csv("../data/requik/inputs/documents.csv", index=False)
corpus_df.head()

100%|██████████| 317/317 [04:48<00:00,  1.10it/s]


,docno,main_content,readability,edu_val
0,RQ_d1,When was the last time the capulin volcano nat...,4.418878,2
1,RQ_d2,Frequently Asked Questions - Capulin Volcano N...,3.050643,2
2,RQ_d3,When was the last time the capulin volcano nat...,4.418878,2
3,RQ_d4,Capulin Volcano National Monument - Britannica...,5.334789,3
4,RQ_d5,Capulin Volcano National Monument - Wikipedia....,4.335093,3


## Creating other versions of ground truth

### relevance + readability

In [22]:
from tqdm import tqdm
import pandas as pd

# nltk.download('all', halt_on_error=False)
corpus = pd.read_csv("../data/requik/inputs/documents.csv")
qrels_df = pd.read_csv("../data/requik/qrels/qrels-relevance-binary.csv")
read_relevance = []
issue_docs = []

for _, row in tqdm(qrels_df.iterrows(), total=len(qrels_df)):
    qid = row["qid"]
    docid = row["docno"]
    relevance = row["relevance"]
    readability = list(corpus.loc[corpus["docno"]==docid]["readability"])[0]
    if readability < 0: # removing query-doc pairs where the readability score is invalid (i.e., the score < 0)
        issue_docs.append(docid)
        continue
    else:
        read_relevance.append([qid, "q0", docid, relevance*int(int(readability)<=4)])

qrels_readability_df = pd.DataFrame(read_relevance, columns=["qid", "q0", "docno", "relevance"])
qrels_readability_df.to_csv("../data/requik/qrels/qrels-readability-binary.csv", index=False)
qrels_readability_df


100%|██████████| 413/413 [00:00<00:00, 3999.75it/s]


,qid,q0,docno,relevance
0,RQq1,q0,RQ_d1,1
1,RQq1,q0,RQ_d2,0
2,RQq1,q0,RQ_d3,0
3,RQq1,q0,RQ_d4,0
4,RQq1,q0,RQ_d5,0
...,...,...,...,...
408,RQq82,q0,RQ_d313,1
409,RQq82,q0,RQ_d314,0
410,RQq82,q0,RQ_d315,0
411,RQq82,q0,RQ_d316,0
